# 빈 배경 오탐 실험 v1 — 데이터 준비와 재학습 전 측정

재학습을 실행하는 셀은 없습니다. 기존 label_fix_v1 데이터에 사용자 확인 배경 35장만 추가한 후보 데이터셋입니다. 평가 16장과 화질 보류 4장은 train에 넣지 않습니다. 실제 장소의 독립성은 아직 확인되지 않았습니다.

GitHub에서 받은 `데이터셋/` 폴더를 사용하거나, 별도 데이터 ZIP을 풀고 아래 경로를 지정하세요. 모델 best.pt는 별도로 준비해야 합니다. 원본 가중치를 변경하지 않습니다. Google Colab에서는 필요할 때 `%pip install -r requirements.txt`로 평가 환경을 설치하세요.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import yaml
PACKAGE = Path.cwd()  # 이 노트북과 Python 파일들이 있는 폴더
PREPARED = PACKAGE / '데이터셋'  # 별도 데이터 ZIP을 사용하면 압축 해제 경로로 변경
WEIGHTS = Path('/workspace/recycling_assets/label_fix_v1/weights/best.pt')
assert (PACKAGE / 'background_experiment.py').is_file()
assert WEIGHTS.is_file(), '실제 label_fix_v1 best.pt가 필요합니다.'
# 압축 해제 위치에 맞춰 새 후보 데이터셋 YAML의 경로만 설정합니다.
data_yaml = PREPARED / 'data/data.yaml'
cfg = yaml.safe_load(data_yaml.read_text())
cfg['path'] = str((PREPARED / 'data').resolve())
data_yaml.write_text(yaml.safe_dump(cfg, allow_unicode=True))
print(json.loads((PREPARED / 'preparation.json').read_text())['audit']['splits'])

## 기존 모델의 배경 오탐 측정

CPU, imgsz=640, confidence=0.25, IoU=0.7로 고정합니다. 16장은 정답 객체가 없으므로 모든 탐지가 오탐입니다. 빈 배경만으로 객체 mAP나 recall을 계산하지 않습니다. 새 결과 폴더가 이미 있으면 덮어쓰지 않고 중단합니다.

In [ ]:
OUT = PREPARED / 'baseline_evaluation_recheck'  # 새 출력 폴더
subprocess.run([sys.executable, str(PACKAGE / 'background_experiment.py'),
                'measure', '--prepared', str(PREPARED), '--weights', str(WEIGHTS),
                '--out', str(OUT)], check=True)
report = json.loads((OUT / 'baseline_negative_evaluation.json').read_text())
{k:v for k,v in report.items() if k != 'details'}

## 다음 비교 실험의 원칙

재학습은 별도 승인 후 진행합니다. 비교할 때 원래 데이터 학습과 배경 35장 추가 학습의 초기 가중치·학습 설정·시드를 같게 하고, 배경 추가 여부만 바꾸세요. 기존 최종 best.pt의 추론 결과는 출발점 기록이며, 그 결과와 새 학습 결과의 차이를 배경 추가의 효과만으로 단정하면 안 됩니다. 두 장의 라벨 수정안도 이번 배경 추가 실험에는 섞지 않았습니다.

배경 오탐과 함께 기존 객체 val 성능, 실제 플라스틱·종이·캔의 미탐 여부를 확인하세요. 오탐이 줄면서 진짜 물체도 못 찾게 되는 경우를 막기 위해서입니다. 평가 16장은 개발용으로 계속 분리하고, 최종 포트폴리오용 사진은 새 장소에서 별도로 확보하세요.